In [1]:
from pathlib import Path
from dmpbridge.pdf.pdfplumber_extractor import save_pdfplumber_outputs

project_root = Path.cwd().parent

pdf_path = project_root / "data" / "raw_pdfs" / "nih_pre_2026" / "sample.pdf"

print("Project root:", project_root)
print("PDF path:", pdf_path)
print("PDF exists:", pdf_path.exists())

blocks = save_pdfplumber_outputs(pdf_path)

print("Number of extracted lines:", len(blocks))

Project root: c:\Users\Nahid\dmpbridge
PDF path: c:\Users\Nahid\dmpbridge\data\raw_pdfs\nih_pre_2026\sample.pdf
PDF exists: True
[2026-05-04 10:14:41] Extracting line-level text with pdfplumber: sample.pdf
[2026-05-04 10:14:41] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_blocks\sample.json
[2026-05-04 10:14:41] Saved extracted text: C:\Users\Nahid\dmpbridge\data\extracted_text\sample.txt
Number of extracted lines: 141


In [2]:
import json
from pathlib import Path
import pandas as pd

from dmpbridge.processing.structure_detector import detect_structure

project_root = Path.cwd().parent
json_path = project_root / "data" / "pdfplumber_blocks" / "sample.json"

with open(json_path, "r", encoding="utf-8") as f:
    blocks = json.load(f)

structured = detect_structure(blocks)

df = pd.DataFrame(structured)

df[["page", "line_order", "text", "avg_font_size", "is_bold", "label"]].head(50)

,page,line_order,text,avg_font_size,is_bold,label
0,1,1,DATA MANAGEMENT AND SHARING PLAN,11.04,True,section
1,1,2,An example from an application proposing to co...,11.04,False,content
2,1,3,If any of the proposed research in the applica...,9.00,False,content
3,1,4,for Data Management and Sharing and requires s...,9.00,False,content
4,1,5,application will generate large-scale genomic ...,9.00,False,content
5,1,6,Refer to the detailed instructions in the appl...,9.00,False,content
6,1,7,The Plan is recommended not to exceed two page...,9.00,False,content
7,1,8,There is no “form page” for the Data Managemen...,9.00,False,content
8,1,9,Element 1: Data Type,11.04,True,section
9,1,10,A. Types and amount of scientific data expecte...,11.04,True,subsection


In [3]:
output_path = project_root / "outputs" / "debug" / "structured_lines.csv"

# make sure folder exists
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False, encoding="utf-8")

print("Saved CSV to:", output_path)

Saved CSV to: c:\Users\Nahid\dmpbridge\outputs\debug\structured_lines.csv


In [4]:
import json
from pathlib import Path
import pandas as pd

from dmpbridge.processing.structure_detector import detect_structure
from dmpbridge.processing.structure_json_builder import save_structure_json

project_root = Path.cwd().parent

input_json = project_root / "data" / "pdfplumber_blocks" / "sample.json"
output_json = project_root / "data" / "structure_json" / "sample_structure.json"

with open(input_json, "r", encoding="utf-8") as f:
    blocks = json.load(f)

structured_blocks = detect_structure(blocks)

structure = save_structure_json(structured_blocks, output_json)

print("Saved:", output_json)
print("Sections:", len(structure["sections"]))

[2026-05-04 10:17:12] Saved structure JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample_structure.json
Saved: c:\Users\Nahid\dmpbridge\data\structure_json\sample_structure.json
Sections: 8


In [5]:
structure["sections"][0]

{'order': 1,
 'title': 'DATA MANAGEMENT AND SHARING PLAN',
 'page': 1,
 'subsections': [],
 'content': [{'text': 'An example from an application proposing to collect clinical and MRI data from human subjects.',
   'page': 1,
   'line_order': 2,
   'label': 'content'},
  {'text': 'If any of the proposed research in the application involves the generation of scientific data, this application is subject to the NIH Policy',
   'page': 1,
   'line_order': 3,
   'label': 'content'},
  {'text': 'for Data Management and Sharing and requires submission of a Data Management and Sharing Plan. If the proposed research in the',
   'page': 1,
   'line_order': 4,
   'label': 'content'},
  {'text': 'application will generate large-scale genomic data, the Genomic Data Sharing Policy also applies and should be addressed in this Plan.',
   'page': 1,
   'line_order': 5,
   'label': 'content'},
  {'text': 'Refer to the detailed instructions in the application guide for developing this plan as well as to a

In [6]:
preview_path = project_root / "outputs" / "debug" / "sample_structure_preview.txt"

lines = []

for section in structure["sections"]:
    lines.append(f"# {section['title']}")

    for item in section.get("content", []):
        lines.append(item["text"])

    for subsection in section.get("subsections", []):
        lines.append(f"\n## {subsection['title']}")

        for item in subsection.get("content", []):
            lines.append(item["text"])

    lines.append("\n")

preview_path.parent.mkdir(parents=True, exist_ok=True)

with open(preview_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("Preview saved:", preview_path)

Preview saved: c:\Users\Nahid\dmpbridge\outputs\debug\sample_structure_preview.txt
